# EX_06 — Introducción a RAG (ejercicios)

**Notebook de referencia:** `notebook/06_Introduccion_RAG.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Plantilla de contexto

Escribe una función `build_prompt(context_chunks, question) -> str` que inserte los pasajes en un delimitador claro (`### Context` / `### Question`).


In [1]:
def build_prompt(context_chunks: list[str], question: str) -> str:
    context = "\n\n".join(
        [f"Passage {i+1}:\n{chunk}" for i, chunk in enumerate(context_chunks)]
    )
    
    prompt = f"""
### Context

{context}

### Question

{question}

### Instructions

Answer the question using only the information provided in the context.
If the answer is not present in the context, say that the information is not available.
"""
    
    return prompt


# Example
context_chunks = [
    "RAG combines retrieval and generation to answer questions using external documents.",
    "The retriever selects relevant passages before the language model generates an answer."
]

question = "What does RAG combine?"

prompt = build_prompt(context_chunks, question)

print(prompt)


### Context

Passage 1:
RAG combines retrieval and generation to answer questions using external documents.

Passage 2:
The retriever selects relevant passages before the language model generates an answer.

### Question

What does RAG combine?

### Instructions

Answer the question using only the information provided in the context.
If the answer is not present in the context, say that the information is not available.



## Actividad 2 — RAG sin LLM (retrieval only)

Con tus chunks del notebook teórico (o texto inventado), recupera top-k y **imprime** el contexto ensamblado sin llamar al generador.


In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Example chunks
chunks = [
    "RAG combines information retrieval with text generation.",
    "The retriever finds relevant documents or passages for a user question.",
    "The generator uses the retrieved context to produce an answer.",
    "Pizza is a popular Italian food made with dough, tomato and cheese."
]

question = "What does the retriever do in a RAG system?"

# Embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Encode chunks and question
chunk_embeddings = model.encode(chunks)
question_embedding = model.encode(question)

# Cosine similarity
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Compute scores
scores = []

for i, chunk_embedding in enumerate(chunk_embeddings):
    score = cosine_similarity(question_embedding, chunk_embedding)
    scores.append((i, score, chunks[i]))

# Retrieve top-k
top_k = 2
top_chunks = sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

# Assemble context
retrieved_context = "\n\n".join(
    [f"Passage {rank+1}:\n{chunk}" for rank, (_, _, chunk) in enumerate(top_chunks)]
)

print("Retrieved context:\n")
print(retrieved_context)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Retrieved context:

Passage 1:
RAG combines information retrieval with text generation.

Passage 2:
The retriever finds relevant documents or passages for a user question.


## Actividad 3 — Fallo de cobertura

Inventa un caso donde la respuesta **no** está en los chunks recuperados y describe en español (markdown) cómo lo detectarías en producción (p. ej. umbral de score, abstención).


_Tu explicación:_


**Pregunta:**  
¿Cuál es la capital de Japón?

Los chunks disponibles solo contienen información sobre RAG, embeddings y recuperación semántica.

En ese caso, aunque el sistema recupere algún chunk, ninguno contiene realmente la respuesta correcta. El modelo podría intentar responder igualmente y generar una alucinación.

Para detectarlo en producción usaría varias estrategias:

- **Umbral mínimo de similitud:** si el score del mejor chunk recuperado está por debajo de un umbral, por ejemplo `0.40`, el sistema no debería generar respuesta.
- **Abstención controlada:** si no hay contexto suficiente, el sistema debe responder: “No tengo información suficiente en los documentos recuperados”.
- **Revisión de cobertura:** comprobar si los chunks recuperados contienen palabras o conceptos relacionados con la pregunta.
- **Logging de scores:** guardar los scores de recuperación para analizar cuándo el retriever está devolviendo contexto débil o irrelevante.

De esta forma se reduce el riesgo de que el sistema RAG responda usando conocimiento inventado en vez de información realmente presente en los documentos.
